# GhostLM — Architecture Exploration

> An open-source cybersecurity-focused language model built from scratch in PyTorch
> Author: Joe Munene | License: Apache 2.0

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')
import torch
from ghostlm import GhostLM, GhostLMConfig, GhostTokenizer

## 2. Model Configurations

Explore the three GhostLM presets

In [ ]:
for preset in ["ghost-tiny", "ghost-small", "ghost-medium"]:
    config = GhostLMConfig.from_preset(preset)
    model = GhostLM(config)
    print(f"{preset:15} | params: {model.num_params():>12,} | size: {config.model_size()}")

## 3. Tokenizer

In [ ]:
tokenizer = GhostTokenizer()
print(tokenizer)
print()

test_texts = [
    "CVE-2023-44487: HTTP/2 Rapid Reset Attack",
    "Buffer overflow in OpenSSL allows remote code execution",
    "Privilege escalation via misconfigured SUID binary",
]

for text in test_texts:
    ids = tokenizer.encode(text)
    decoded = tokenizer.decode(ids)
    print(f"Text:    {text}")
    print(f"Tokens:  {len(ids)}")
    print(f"Decoded: {decoded}")
    print()

## 4. Forward Pass

In [ ]:
config = GhostLMConfig.from_preset("ghost-tiny")
config.vocab_size = 50261
config.context_length = 64
model = GhostLM(config)
model.eval()

batch_size = 2
seq_len = 32
x = torch.randint(0, config.vocab_size, (batch_size, seq_len))
y = torch.randint(0, config.vocab_size, (batch_size, seq_len))

with torch.no_grad():
    logits, loss = model(x, targets=y)

print(f"Input shape:   {x.shape}")
print(f"Logits shape:  {logits.shape}")
print(f"Loss:          {loss.item():.4f}")
print(f"Expected loss (random): {torch.log(torch.tensor(config.vocab_size)).item():.4f}")

## 5. Text Generation (Untrained)

Note: outputs are random since model is not yet trained

In [ ]:
tokenizer = GhostTokenizer()
prompt = "A SQL injection attack works by"
ids = tokenizer.encode(prompt)
input_tensor = torch.tensor(ids, dtype=torch.long).unsqueeze(0)

print(f"Prompt: {prompt}")
print(f"Prompt tokens: {len(ids)}")
print()

with torch.no_grad():
    output = model.generate(input_tensor, max_new_tokens=50, temperature=0.8, top_k=50)

generated_text = tokenizer.decode(output[0].tolist())
print(f"Generated (untrained — random output expected):")
print(generated_text)

## 6. Architecture Summary

In [ ]:
config = GhostLMConfig.from_preset("ghost-small")
print(repr(config))

## 7. Next Steps

- Run `python data/collect.py` to download cybersecurity training data
- Run `python scripts/train.py --preset ghost-tiny` to train the small model
- Run `python scripts/benchmark.py` to compare against GPT-2 baseline
- Run `python scripts/generate.py --checkpoint checkpoints/best_model.pt --prompt "Explain CVE-2023-44487"`